## 체크포인트 작동 확인하기


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

LANGSMITH_TRACING=os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT=os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

### 그래프 스테이트와 노드 정의

In [7]:
import operator
from typing import Annotated, Any
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 그래프의 스테이트 정의
class State(BaseModel):
    query: str # 사용자 입력
    messages : Annotated[list[BaseMessage], operator.add] = Field(default=[]) # 사용자와 gpt 간의 대화 기록을 저장

# 메시지를 추가하는 노드 함수
def add_message(state: State) -> dict[str, Any]:
    additional_messages = []
    if not state.messages:
        additional_messages.append(SystemMessage(content="당신은 최소한의 응답을 하는 대화 에이전트입니다."))
    additional_messages.append(HumanMessage(content=state.query))
    return {"messages" : additional_messages}

# LLM응답을 추가하는 노드 함수
def llm_response(state: State) -> dict[str, Any]:
    llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
    ai_message = llm.invoke(state.messages)
    return {"messages": [ai_message]}

### 체크포인트 내용을 표시하는 함수 정의

In [8]:
from pprint import pprint
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.base import BaseCheckpointSaver

def print_checkpoint_dump(checkpointer: BaseCheckpointSaver, config: RunnableConfig):
    checkpoint_tuple = checkpointer.get_tuple(config)
    print("체크포인트 데이터: ")
    pprint(checkpoint_tuple.checkpoint)
    print("\n메타데이터: ")
    pprint(checkpoint_tuple.metadata)

### 그래프 정의 및 컴파일

In [9]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# 그래프 설정
graph = StateGraph(State)
graph.add_node("add_message", add_message)
graph.add_node("llm_response", llm_response)
graph.set_entry_point("add_message")
graph.add_edge("add_message", "llm_response")
graph.add_edge("llm_response", END)

checkpointer = MemorySaver()

compiled_graph = graph.compile(checkpointer=checkpointer)

In [10]:
config = {"configurable" : {"thread_id" : "example-1"}}
user_query = State(query="제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요")
first_response = compiled_graph.invoke(user_query, config)
first_response

{'query': '제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요',
 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요', additional_kwargs={}, response_metadata={}),
  AIMessage(content='기억할게요. 당신이 좋아하는 것은 찹쌀떡입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 42, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DkpoWFPCyxjVMLugtQJepgOCZiY7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7386-224f-72f2-a735-f57667c7737d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens'

- 이제 체크포인터에 대해 list 함수를 이용해 체크포인트 정보가 어떻게 변했는지 살펴보자

In [12]:
for checkpoint in checkpointer.list(config):
    pprint(checkpoint)

Deserializing unregistered type __main__.State from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'State')]


CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b52b-c09a-613e-8002-25d9ee9153c9'}}, checkpoint={'v': 4, 'ts': '2026-05-29T11:37:12.538958+00:00', 'id': '1f15b52b-c09a-613e-8002-25d9ee9153c9', 'channel_versions': {'__start__': '00000000000000000000000000000002.0.37479161074003686', 'query': '00000000000000000000000000000002.0.37479161074003686', 'messages': '00000000000000000000000000000004.0.6223051335493586', 'branch:to:add_message': '00000000000000000000000000000003.0.9637372050461831', 'branch:to:llm_response': '00000000000000000000000000000004.0.6223051335493586'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000001.0.16338775389860893'}, 'add_message': {'branch:to:add_message': '00000000000000000000000000000002.0.37479161074003686'}, 'llm_response': {'branch:to:llm_response': '00000000000000000000000000000003.0.9637372050461831'}}, 'updated_channels': ['messages'], 'channel_v

In [13]:
print_checkpoint_dump(checkpointer, config)

체크포인트 데이터: 
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='기억할게요. 당신이 좋아하는 것은 찹쌀떡입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 42, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DkpoWFPCyxjVMLugtQJepgOCZiY7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7386-224f-72f2-a735-f57667c7737d-0', tool_calls=[], invalid_tool_calls=[

In [14]:
user_query = State(query="제가 좋아하는 것이 뭔지 기억하세요?")
second_response = compiled_graph.invoke(user_query, config)
second_response

{'query': '제가 좋아하는 것이 뭔지 기억하세요?',
 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요', additional_kwargs={}, response_metadata={}),
  AIMessage(content='기억할게요. 당신이 좋아하는 것은 찹쌀떡입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 42, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DkpoWFPCyxjVMLugtQJepgOCZiY7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7386-224f-72f2-a735-f57667c7737d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 22,

In [15]:
for checkpoint in checkpointer.list(config):
    print(checkpoint)

Deserializing unregistered type __main__.State from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'State')]
Deserializing unregistered type __main__.State from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'State')]


CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b532-735e-62b0-8006-d2065fd27179'}}, checkpoint={'v': 4, 'ts': '2026-05-29T11:40:12.345196+00:00', 'id': '1f15b532-735e-62b0-8006-d2065fd27179', 'channel_versions': {'__start__': '00000000000000000000000000000006.0.3225368560853702', 'query': '00000000000000000000000000000006.0.3225368560853702', 'messages': '00000000000000000000000000000008.0.23380036641243696', 'branch:to:add_message': '00000000000000000000000000000007.0.9571350001802218', 'branch:to:llm_response': '00000000000000000000000000000008.0.23380036641243696'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000005.0.38085632368526134'}, 'add_message': {'branch:to:add_message': '00000000000000000000000000000006.0.3225368560853702'}, 'llm_response': {'branch:to:llm_response': '00000000000000000000000000000007.0.9571350001802218'}}, 'updated_channels': ['messages'], 'channel_va

In [16]:
print_checkpoint_dump(checkpointer, config)

체크포인트 데이터: 
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찹쌀떡입니다. 기억해주세요', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='기억할게요. 당신이 좋아하는 것은 찹쌀떡입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 42, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DkpoWFPCyxjVMLugtQJepgOCZiY7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7386-224f-72f2-a735-f57667c7737d-0', tool_calls=[], invalid_tool_calls=[